<a href="https://colab.research.google.com/github/chamarairesh1982/ml-cybersecurity/blob/main/section4_ransomware/section4_ransomware.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Saved-run review and current protocol

Version 2 development revision: fitting before July 2016; stopping July–August; selection September–October; policy November–December. Keep all fitting positives, uniformly sample background within a 220,000-row budget, and combine inverse inclusion with original-population class balancing exactly once. Prior uses inverse inclusion only. Validate on complete 2014, 2015 and first-half-2016 forward cohorts. Selection/stopping/policy remain uncapped. The 2017–2018 test period overlaps inspected historical data, so RUN_HISTORICAL_TEST=False by default. No fresh holdout is claimed.

The historical findings and limitations are recorded in REPORT.md. The configuration and disjoint boundaries above govern this revision. Run in a fresh Colab runtime; independently measured improvement remains unverified.


# Section 4 — ransomware-linked payment screening
A reproducible learning experiment: can daily Bitcoin graph measurements rank **previously unseen addresses** carrying a known ransomware label above the provider's background sample?

This is payment-address screening after a complete daily graph window is available. It does not detect file encryption on a computer. Provider `white` means **not known to be ransomware**, rather than verified benign. All metrics refer to these retrospective corpus labels.

Run in a fresh Colab runtime, in order. Data stays in Colab memory; aggregate evidence stays in `/content/section4_results/v2_development`. No Drive mount, credentials or malware execution is needed. Classical models use CPU. The optional numerical neural comparator supports TensorFlow CPU/GPU/TPU strategies; accelerator execution remains to be verified in Colab.


## 1. Question, sources and protocol
The workflow follows the learning concepts of data preparation/EDA, feature engineering, supervised learning, neural networks, ensembles and model evaluation. NLP and clustering are not forced onto numeric payment features.

Use [UCI BitcoinHeist](https://archive.ics.uci.edu/dataset/526/bitcoinheistransomwareaddressdataset), DOI [10.24432/C5BG8V](https://doi.org/10.24432/C5BG8V), CC BY 4.0. See the [original research](https://arxiv.org/abs/1906.07852). UCI reports 2,916,697 daily observations from 2009–2018, six graph measurements, address/year/day metadata and a label. Background addresses were capped at 1,000 per day and small transfers were filtered. Sample precision cannot be transferred to live Bitcoin traffic.

**Revised chronological boundaries:** fitting through June 2016; neural stopping July–August 2016; selection September–October 2016; policy November–December 2016. Forward CV validates on 2014, 2015 and January–June 2016. Inspected 2017–2018 replay is optional historical evidence. No random fallback if class support fails.

Retain each address's earliest eligible observation and exclude addresses observed in any earlier period, including dirty earlier records with usable dates. Address, calendar fields and family labels never enter predictors. This changes the population to newly observed addresses. Addresses are not verified independent actors or campaigns. Labels were assembled retrospectively; their historical availability is unknown. An unseen family is not evidence of a previously unknown attack.

Select by average precision (AP), with fitting time breaking exact ties. Freeze the model before using the policy partition to set a threshold targeting at most 1% observed alerts on labelled background. This is an empirical sample budget, not a statistical deployment guarantee. Test once, then require a new independent holdout for subsequent tuning.


In [ ]:
import os, io, json, hashlib, zipfile, time, platform, warnings
from pathlib import Path
os.environ.setdefault("OMP_NUM_THREADS", "2")
os.environ.setdefault("TF_NUM_INTRAOP_THREADS", "2")
os.environ.setdefault("TF_NUM_INTEROP_THREADS", "2")
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
import sklearn
from IPython.display import display
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (average_precision_score, roc_auc_score, precision_recall_curve,
                             confusion_matrix, ConfusionMatrixDisplay)
from sklearn.inspection import permutation_importance
from threadpoolctl import threadpool_limits
SEED = 42
MAX_FIT_ADDRESSES = 220_000  # keep all fitting positives; sample background only
MAX_CV_EVAL = None  # complete forward validation; no rare-positive loss
RUN_NEURAL = True
NEURAL_SEEDS = (42, 123)
EPOCHS = 25
BATCH_SIZE = 1024
BACKGROUND_BUDGET = 0.01
BOOTSTRAPS = 300
URL = "https://archive.ics.uci.edu/static/public/526/bitcoinheistransomwareaddressdataset.zip"
# No independently verified publisher checksum was found. Record runtime SHA-256;
# optionally set a checksum established independently for a strictly repeated run.
EXPECTED_SHA256 = None
BASE = ["length", "weight", "count", "looped", "neighbors", "income"]
DERIVED = ["income_per_neighbor", "weight_per_count", "loop_share"]
FEATURES = BASE + DERIVED
SCHEMA = ["address", "year", "day"] + BASE + ["label"]
CUTS = [pd.Timestamp(x) for x in ("2016-07-01", "2016-09-01", "2016-11-01", "2017-01-01")]
RESULTS = Path("/content/section4_results/v2_development")
RESULTS.mkdir(parents=True, exist_ok=True)
np.random.seed(SEED)
versions = {"python": platform.python_version(), "numpy": np.__version__,
            "pandas": pd.__version__, "sklearn": sklearn.__version__}
print(versions)
print("Fit cap:", MAX_FIT_ADDRESSES, "| Neural:", RUN_NEURAL)

RUN_HISTORICAL_TEST = False  # Inspected saved benchmark; opt-in replay only.
EXPERIMENT_VERSION = "section4-v2-development"


## 2. Acquire and audit
The roughly 111 MB archive is downloaded once into runtime memory. It contains the whole corpus; test-period rows are not retained, profiled or used until the model and threshold are frozen. Parsing in chunks controls temporary memory. The original UCI URL is fixed, but is not an immutable content pin. HTTPS, ZIP CRC checks, schema checks and a recorded runtime SHA-256 provide audit evidence; a recorded hash alone does not authenticate an unknown download.

Address strings remain internal only for grouping. Displays and exports contain aggregate counts, never addresses or transaction rows. Calendar checks support leap years. Missing/nonfinite measurements are marked for training-only imputation; negative measurements, missing targets/addresses and invalid calendar records are excluded. Duplicate or conflicting address-day observations are audited within the opened period; conflicting copies are excluded. No future labels are used to clean development.


In [ ]:
def calendar_dates(frame):
    years = pd.to_numeric(frame["year"], errors="coerce")
    days = pd.to_numeric(frame["day"], errors="coerce")
    valid = (years.between(2009, 2018) & days.between(1, 366)
             & years.eq(np.floor(years)) & days.eq(np.floor(days)))
    starts = pd.to_datetime(years.where(valid).astype("Int64").astype("string") + "-01-01", errors="coerce")
    dates = starts + pd.to_timedelta(days.where(valid) - 1, unit="D")
    return dates.where(valid & dates.dt.year.eq(years))

def clean_period(raw):
    audit = {"raw_period_rows": len(raw)}
    d = raw.copy()
    d["date"] = calendar_dates(d)
    d["address"] = d["address"].astype("string").str.strip()
    d["family"] = d["label"].astype("string").str.strip()
    usable = d["date"].notna() & d["address"].notna() & d["address"].ne("") & d["family"].notna() & d["family"].ne("")
    for name in BASE:
        d[name] = pd.to_numeric(d[name], errors="coerce").replace([np.inf, -np.inf], np.nan)
    audit["missing_feature_cells"] = int(d[BASE].isna().sum().sum())
    usable &= ~d[BASE].lt(0).any(axis=1)
    audit["invalid_rows"] = int((~usable).sum())
    d = d.loc[usable].copy()
    d["y"] = d["family"].ne("white").astype("int8")
    key = ["address", "date"]
    # Conflicting features/families at the same address-day have no safe ordering.
    unique = d.drop_duplicates(key + BASE + ["family"])
    conflicts = unique.groupby(key, sort=False, observed=True).size()
    bad = conflicts[conflicts > 1].index
    ambiguous = pd.MultiIndex.from_frame(d[key]).isin(bad)
    audit["ambiguous_address_days"] = len(bad)
    audit["ambiguous_rows"] = int(ambiguous.sum())
    d = d.loc[~ambiguous].copy()
    before = len(d)
    d = d.drop_duplicates(key + BASE + ["family"])
    audit["duplicate_copies"] = before - len(d)
    before = len(d)
    d = d.sort_values("date", kind="stable").drop_duplicates("address", keep="first")
    audit["later_observations_removed"] = before - len(d)
    # Row-local transformations use neither labels nor learned statistics.
    d["income_per_neighbor"] = d["income"] / (1 + d["neighbors"])
    d["weight_per_count"] = d["weight"] / (1 + d["count"])
    d["loop_share"] = d["looped"] / (1 + d["count"])
    d[FEATURES] = d[FEATURES].replace([np.inf, -np.inf], np.nan)
    d = d[["address", "date", "family", "y"] + FEATURES].reset_index(drop=True)
    audit["retained_addresses"] = len(d)
    return d, audit

def read_period(payload, start, end):
    pieces, seen, counts = [], set(), {"parsed_rows": 0, "invalid_calendar_rows": 0}
    with zipfile.ZipFile(io.BytesIO(payload)) as archive:
        names = [n for n in archive.namelist() if n.split("/")[-1] == "BitcoinHeistData.csv"]
        if len(names) != 1:
            raise ValueError("Expected exactly one BitcoinHeistData.csv")
        with archive.open(names[0]) as stream:
            for chunk in pd.read_csv(stream, chunksize=100_000, dtype={"address": "string", "label": "string"}):
                if set(chunk.columns) != set(SCHEMA):
                    raise ValueError("Unexpected dataset schema")
                counts["parsed_rows"] += len(chunk)
                dates = calendar_dates(chunk)
                counts["invalid_calendar_rows"] += int(dates.isna().sum())
                prior = chunk.loc[dates < start, "address"].dropna().str.strip()
                seen.update(prior[prior.ne("")].tolist())
                keep = dates.ge(start) & dates.lt(end)
                if keep.any():
                    pieces.append(chunk.loc[keep, SCHEMA].copy())
    if not pieces:
        raise ValueError("No rows in declared period; do not change cutoffs after testing")
    raw = pd.concat(pieces, ignore_index=True)
    counts["opened_period_rows"] = len(raw)
    counts["prior_address_exclusions"] = int(raw["address"].astype("string").str.strip().isin(seen).sum())
    raw = raw.loc[~raw["address"].astype("string").str.strip().isin(seen)]
    raw_dates = calendar_dates(raw)
    first_dates = raw_dates.groupby(raw["address"].astype("string").str.strip()).transform("min")
    earliest = raw_dates.eq(first_dates)
    counts["later_observation_rows_excluded_before_cleaning"] = int((~earliest).sum())
    # Invalid/ambiguous first observations never promote an address into a later cohort.
    cleaned, audit = clean_period(raw.loc[earliest])
    counts.update(audit)
    return cleaned, counts

def require_binary(d, name):
    counts = d["y"].value_counts().reindex([0, 1], fill_value=0)
    if counts.min() < 10:
        raise ValueError(f"{name}: insufficient class support {counts.to_dict()}; stop, no random fallback")

def cap_rows(d, maximum, seed=SEED):
    # One row per address: uniform address sampling, never class-dependent.
    return (d if maximum is None else d.sample(min(len(d), maximum), random_state=seed)).sort_values("date").reset_index(drop=True)

def sample_training(d, maximum=MAX_FIT_ADDRESSES, seed=SEED):
    require_binary(d, "training source")
    positive = d.loc[d.y == 1]
    background = d.loc[d.y == 0]
    slots = maximum - len(positive)
    if slots < 100:
        raise ValueError("Fitting cap cannot retain every positive plus background; increase the declared cap")
    sampled_background = background.sample(min(len(background), slots), random_state=seed)
    result = pd.concat([positive, sampled_background]).sort_values("date").reset_index(drop=True)
    inclusion = len(sampled_background) / len(background)
    result["_population_weight"] = np.where(result.y == 0, 1 / inclusion, 1.)
    counts = d.y.value_counts()
    # Combine inverse inclusion and original-population class balancing exactly once.
    balanced = result["_population_weight"].to_numpy() * np.where(result.y == 0,
                  len(d) / (2 * counts[0]), len(d) / (2 * counts[1]))
    result["_balanced_weight"] = balanced / balanced.mean()
    assert int(result.y.sum()) == len(positive)
    return result

def download_archive():
    started = time.perf_counter()
    with requests.get(URL, stream=True, timeout=(30, 180)) as response:
        response.raise_for_status()
        buf = io.BytesIO()
        for piece in response.iter_content(1024 * 1024):
            buf.write(piece)
            if buf.tell() > 160_000_000:
                raise ValueError("Archive exceeds expected size envelope")
        data = buf.getvalue()
    sha = hashlib.sha256(data).hexdigest()
    if EXPECTED_SHA256 is not None and sha != EXPECTED_SHA256:
        raise ValueError("Archive SHA-256 mismatch")
    if not zipfile.is_zipfile(io.BytesIO(data)):
        raise ValueError("Response is not a ZIP archive")
    return data, {"url": URL, "sha256": sha, "bytes": len(data),
                  "download_seconds": time.perf_counter() - started,
                  "externally_verified_checksum": EXPECTED_SHA256 is not None}


In [ ]:
archive_bytes, provenance = download_archive()
development, development_audit = read_period(archive_bytes, pd.Timestamp("2009-01-01"), CUTS[3])
fit_pool = development.loc[development.date < CUTS[0]].copy()
stopping = development.loc[development.date.ge(CUTS[0]) & development.date.lt(CUTS[1])].copy()
selection = development.loc[development.date.ge(CUTS[1]) & development.date.lt(CUTS[2])].copy()
policy = development.loc[development.date.ge(CUTS[2])].copy()
fitting = sample_training(fit_pool, MAX_FIT_ADDRESSES)
parts = {"fit_pool": fit_pool, "fitting": fitting, "stopping": stopping,
         "selection": selection, "policy": policy}
for name, part in parts.items():
    require_binary(part, name)
for a, b in (("fitting", "stopping"), ("stopping", "selection"), ("selection", "policy")):
    assert parts[a].date.max() < parts[b].date.min()
    assert not set(parts[a].address).intersection(parts[b].address)
display(pd.DataFrame([{ "partition": name, "addresses": len(d),
                       "labelled_ransomware": int(d.y.sum()), "labelled_background": int((d.y == 0).sum()),
                       "start": str(d.date.min().date()), "end": str(d.date.max().date())}
                      for name, d in parts.items()]))
display(pd.Series(development_audit, name="development_audit"))
display(pd.Series(provenance, name="source_identity"))
print("Background cap exclusions:", len(fit_pool) - len(fitting), "| positives retained:", int(fitting.y.sum()), "of", int(fit_pool.y.sum()))


## 3. Training-only EDA and feature preparation
Inspect fitting data only: support, missingness, skew and relationships. Six provider measurements describe daily graph paths and income. Three fixed ratios probe income/neighbor and merging behaviour; a core-feature ablation tests whether they help.

Median imputation, log1p and standardisation fit inside each training fold. Log1p reduces heavy right skew without clipping legitimate large values. Tree candidates share the same preparation for a controlled comparison. Labels, address and date are excluded by explicit column selection. Class weights change optimisation, not the prevalence of held-out partitions; resulting outputs are ranking scores, not calibrated payment probabilities.


In [ ]:
display(fitting[FEATURES].describe(percentiles=[.01, .5, .99]).T)
display(fitting.groupby("family", observed=True).size().sort_values(ascending=False).rename("fit_addresses").head(20))
display(fitting[FEATURES].isna().mean().rename("missing_fraction"))
np.log1p(fitting[BASE]).hist(bins=40, figsize=(12, 7))
plt.suptitle("Fitting partition only: log1p graph measurements")
plt.tight_layout()
plt.show()
fig, ax = plt.subplots(figsize=(8, 6))
corr = np.log1p(fitting[FEATURES]).corr()
im = ax.imshow(corr, vmin=-1, vmax=1, cmap="coolwarm")
ax.set_xticks(range(len(FEATURES)), FEATURES, rotation=90)
ax.set_yticks(range(len(FEATURES)), FEATURES)
fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()


In [ ]:
def preprocessing():
    return Pipeline([("impute", SimpleImputer(strategy="median", keep_empty_features=True)),
                     ("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
                     ("scale", StandardScaler())])

def candidate_specs():
    return {
        "prior": (FEATURES, [DummyClassifier(strategy="prior")]),
        "logistic": (FEATURES, [LogisticRegression(C=c, class_weight=None, max_iter=500, random_state=SEED)
                               for c in (.1, 1.)]),
        "forest": (FEATURES, [RandomForestClassifier(n_estimators=160, max_depth=depth, min_samples_leaf=8,
                                max_features="sqrt", class_weight=None, n_jobs=2, random_state=SEED)
                              for depth in (10, 18)]),
        "boosting": (FEATURES, [HistGradientBoostingClassifier(max_iter=160, learning_rate=.07, max_leaf_nodes=leaves,
                                    min_samples_leaf=40, l2_regularization=2., early_stopping=False,
                                    class_weight="balanced", random_state=SEED) for leaves in (15, 31)]),
        "boosting_core": (BASE, [HistGradientBoostingClassifier(max_iter=160, learning_rate=.07, max_leaf_nodes=15,
                                    min_samples_leaf=40, l2_regularization=2., early_stopping=False,
                                    class_weight="balanced", random_state=SEED)])
    }

def forward_folds(pool):
    folds = []
    for year in (2014, 2015, 2016):
        train = pool.loc[pool.date < pd.Timestamp(f"{year}-01-01")]
        valid = pool.loc[pool.date.ge(pd.Timestamp(f"{year}-01-01")) & pool.date.lt(min(pd.Timestamp(f"{year+1}-01-01"), CUTS[0]))]
        train, valid = sample_training(train, MAX_FIT_ADDRESSES), cap_rows(valid, MAX_CV_EVAL)
        require_binary(train, f"CV before {year}")
        require_binary(valid, f"CV {year}")
        assert train.date.max() < valid.date.min()
        assert not set(train.address).intersection(valid.address)
        folds.append((year, train, valid))
    return folds

def ranked_scores(bundle, d):
    with threadpool_limits(limits=2):
        if bundle["kind"] == "neural":
            values = bundle["prep"].transform(d[bundle["columns"]]).astype("float32")
            score = bundle["model"].predict(values, batch_size=BATCH_SIZE, verbose=0).ravel()
        else:
            score = bundle["model"].predict_proba(d[bundle["columns"]])[:, 1]
    score = np.asarray(score, dtype=float)
    if len(score) != len(d) or not np.isfinite(score).all():
        raise ValueError("Nonfinite or misaligned scores")
    return score


## 4. Forward validation, baselines and ensembles
Use identical expanding-window training folds, each retaining all fitting positives with uniformly sampled background and audited weights and freshly fitted preprocessing. Forward validation, stopping, selection and policy are uncapped. Report fold support and training/validation AP.

For each family select the configuration with the greatest mean AP minus its fold standard deviation, a declared stability heuristic rather than a significance test. The prior establishes prevalence-level AP; logistic regression supplies a simple linear baseline. Bagging (forest), boosting and a core-feature ablation test nonlinearity and feature engineering. No automatic random early-stopping split in boosting crosses time boundaries.


In [ ]:
folds = forward_folds(fit_pool)
display(pd.DataFrame([{"validation_year": year, "train": len(tr), "validation": len(va),
                      "train_positive": int(tr.y.sum()), "validation_positive": int(va.y.sum())}
                     for year, tr, va in folds]))
cv_rows, bundles, fit_seconds = [], {}, {}
for name, (columns, models) in candidate_specs().items():
    configurations = []
    for config_id, model in enumerate(models):
        aps = []
        for year, tr, va in folds:
            pipeline = Pipeline([("prep", preprocessing()), ("model", clone(model))])
            started = time.perf_counter()
            with threadpool_limits(limits=2):
                pipeline.fit(tr[columns], tr.y, model__sample_weight=tr["_population_weight" if name=="prior" else "_balanced_weight"])
                train_ap = average_precision_score(tr.y, pipeline.predict_proba(tr[columns])[:, 1], sample_weight=tr["_population_weight"])
                ap = average_precision_score(va.y, pipeline.predict_proba(va[columns])[:, 1])
            aps.append(ap)
            cv_rows.append({"family": name, "configuration": config_id, "year": year,
                            "train_ap": train_ap, "validation_ap": ap,
                            "seconds": time.perf_counter() - started})
        configurations.append((float(np.mean(aps) - np.std(aps)), config_id))
    _, chosen = max(configurations, key=lambda item: (item[0], -item[1]))
    pipeline = Pipeline([("prep", preprocessing()), ("model", clone(models[chosen]))])
    started = time.perf_counter()
    with threadpool_limits(limits=2):
        pipeline.fit(fitting[columns], fitting.y, model__sample_weight=fitting["_population_weight" if name=="prior" else "_balanced_weight"])
    fit_seconds[name] = time.perf_counter() - started
    bundles[name] = {"kind": "classical", "columns": columns, "model": pipeline,
                     "configuration": chosen}
    print(name, "selected configuration", chosen, "fit seconds", round(fit_seconds[name], 2))
cv_table = pd.DataFrame(cv_rows)
display(cv_table)


## 5. Optional multilayer neural comparator
Use the same capped fitting addresses, numerical preprocessing fitted only there, weighted binary cross-entropy, dropout and L2 regularisation. Two seeds reveal some optimiser variability. Early stopping uses the separate July–August-2016 stopping partition's PR-AUC; model selection uses the later selection partition. TensorFlow's binned PR-AUC is only the stopping proxy; final selection uses scikit-learn AP.

CPU is often sufficient for nine inputs; a TPU need not make this small task faster. Strategy creation occurs once. Explicit accelerator initialisation failures raise an error rather than silently claiming accelerator use. See [TensorFlow distribution documentation](https://www.tensorflow.org/guide/distributed_training). Set `RUN_NEURAL=False` before Run all to compare CPU models alone; record this changed candidate set.


In [ ]:
def runtime_strategy(tf):
    if os.environ.get("COLAB_TPU_ADDR") or os.environ.get("TPU_NAME"):
        resolver = tf.distribute.cluster_resolver.TPUClusterResolver()
        tf.config.experimental_connect_to_cluster(resolver)
        tf.tpu.experimental.initialize_tpu_system(resolver)
        return tf.distribute.TPUStrategy(resolver)
    # Current Colab may expose a TPU without these legacy variables.
    try:
        resolver = tf.distribute.cluster_resolver.TPUClusterResolver(tpu="")
    except ValueError:
        resolver = None
    if resolver is not None:
        tf.config.experimental_connect_to_cluster(resolver)
        tf.tpu.experimental.initialize_tpu_system(resolver)
        return tf.distribute.TPUStrategy(resolver)
    if tf.config.list_physical_devices("GPU"):
        return tf.distribute.MirroredStrategy()
    return tf.distribute.get_strategy()

def neural_fit(tf, strategy, train, stop, seed):
    tf.keras.utils.set_random_seed(seed)
    prep = preprocessing()
    x = prep.fit_transform(train[FEATURES]).astype("float32")
    xv = prep.transform(stop[FEATURES]).astype("float32")
    y, yv = train.y.to_numpy(dtype="float32"), stop.y.to_numpy(dtype="float32")
    counts = np.bincount(y.astype(int), minlength=2)
    if counts.min() == 0:
        raise ValueError("Neural fit needs both classes")
    weights = {i: float(len(y) / (2 * counts[i])) for i in (0, 1)}
    with strategy.scope():
        model = tf.keras.Sequential([
            tf.keras.layers.Input(shape=(len(FEATURES),)),
            tf.keras.layers.Dense(64, activation="relu", kernel_regularizer=tf.keras.regularizers.l2(.001)),
            tf.keras.layers.Dropout(.25),
            tf.keras.layers.Dense(32, activation="relu", kernel_regularizer=tf.keras.regularizers.l2(.001)),
            tf.keras.layers.Dropout(.15),
            tf.keras.layers.Dense(1, activation="sigmoid")])
        model.compile(optimizer=tf.keras.optimizers.Adam(.001),
                      loss="binary_crossentropy", metrics=[tf.keras.metrics.AUC(curve="PR", name="pr_auc")])
    batch = max(BATCH_SIZE, 128 * strategy.num_replicas_in_sync)
    # Arrays are numerical; no strings are passed to an accelerator.
    history = model.fit(x, y, validation_data=(xv, yv), sample_weight=train["_balanced_weight"].to_numpy(dtype="float32"),
                        epochs=EPOCHS, batch_size=batch, verbose=2,
                        callbacks=[tf.keras.callbacks.EarlyStopping(monitor="val_pr_auc", mode="max",
                                                                    patience=4, restore_best_weights=True),
                                   tf.keras.callbacks.TerminateOnNaN()])
    if not all(np.isfinite(v).all() for v in history.history.values()):
        raise ValueError("Nonfinite neural training history")
    return {"kind": "neural", "columns": FEATURES, "model": model, "prep": prep}, history.history


In [ ]:
neural_histories, accelerator = {}, "neural disabled"
if RUN_NEURAL:
    import tensorflow as tf
    versions["tensorflow"] = tf.__version__
    strategy = runtime_strategy(tf)
    accelerator = type(strategy).__name__
    print("Actual runtime strategy:", accelerator, "replicas:", strategy.num_replicas_in_sync)
    for seed in NEURAL_SEEDS:
        name = f"neural_seed_{seed}"
        started = time.perf_counter()
        bundles[name], neural_histories[name] = neural_fit(tf, strategy, fitting, stopping, seed)
        fit_seconds[name] = time.perf_counter() - started
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for name, history in neural_histories.items():
        for axis, metric in zip(axes, ("loss", "pr_auc")):
            axis.plot(history[metric], label=name + " fit")
            axis.plot(history["val_" + metric], linestyle="--", label=name + " stopping")
            axis.set_title(metric)
            axis.set_xlabel("Epoch")
    for axis in axes:
        axis.legend(fontsize=7)
    plt.tight_layout()
    plt.show()


## 6. Choose once, freeze, then calibrate policy
Compare every candidate on the identical September–October-2016 selection cohort by exact AP. The fixed equal-score ensemble averages forest and boosting outputs without learning blend weights; score magnitudes are not calibrated probabilities. Neural seeds are separate candidates, not hidden retries. Record the gap above background prevalence, ROC-AUC and fitting cost.

After choosing a winner, the policy partition is opened for the winner alone. Its background scores determine a conservative empirical 1% threshold; tied scores are excluded. Do not re-select models or refit after calibration. Low policy recall is a finding to report, not a reason to optimise on the final test.


In [ ]:
selection_scores, selection_rows = {}, []
for name, bundle in bundles.items():
    started = time.perf_counter()
    score = ranked_scores(bundle, selection)
    selection_scores[name] = score
    selection_rows.append({"candidate": name, "ap": float(average_precision_score(selection.y, score)),
                           "roc_auc": float(roc_auc_score(selection.y, score)),
                           "fit_seconds": fit_seconds[name], "score_seconds": time.perf_counter() - started})
selection_scores["fixed_ensemble"] = (selection_scores["forest"] + selection_scores["boosting"]) / 2
selection_rows.append({"candidate": "fixed_ensemble",
                       "ap": float(average_precision_score(selection.y, selection_scores["fixed_ensemble"])),
                       "roc_auc": float(roc_auc_score(selection.y, selection_scores["fixed_ensemble"])),
                       "fit_seconds": fit_seconds["forest"] + fit_seconds["boosting"], "score_seconds": None})
selection_table = pd.DataFrame(selection_rows).sort_values(["ap", "fit_seconds"], ascending=[False, True])
winner = str(selection_table.iloc[0]["candidate"])
FROZEN = True
def winner_scores(d):
    if winner == "fixed_ensemble":
        return (ranked_scores(bundles["forest"], d) + ranked_scores(bundles["boosting"], d)) / 2
    return ranked_scores(bundles[winner], d)

def empirical_threshold(background, budget=BACKGROUND_BUDGET):
    score = np.asarray(background, dtype=float)
    if len(score) == 0 or not np.isfinite(score).all() or not 0 <= budget < 1:
        raise ValueError("Invalid calibration scores or budget")
    allowed = int(np.floor(len(score) * budget))
    # Above score at descending index 'allowed': at most 'allowed' sample alerts.
    return float(np.nextafter(np.sort(score)[::-1][allowed], np.inf))

def operating_metrics(y, score, threshold):
    y = np.asarray(y, dtype=int)
    pred = np.asarray(score) >= threshold
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    ratio = lambda a, b: float(a / b) if b else None
    return {"n": int(len(y)), "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
            "accuracy": ratio(tp + tn, len(y)), "f1": ratio(2 * tp, 2 * tp + fp + fn),
            "precision": ratio(tp, tp + fp), "recall": ratio(tp, tp + fn),
            "background_alerts_per_1000": ratio(1000 * fp, fp + tn),
            "background_alert_rate": ratio(fp, fp + tn), "alert_fraction": ratio(tp + fp, len(y)),
            "ap": float(average_precision_score(y, score)) if len(np.unique(y)) == 2 else None,
            "roc_auc": float(roc_auc_score(y, score)) if len(np.unique(y)) == 2 else None}

display(selection_table)
print("Selection positive prevalence:", selection.y.mean(), "| frozen winner:", winner)
policy_score = winner_scores(policy)
threshold = empirical_threshold(policy_score[policy.y.to_numpy() == 0])
policy_metrics = operating_metrics(policy.y, policy_score, threshold)
assert policy_metrics["background_alert_rate"] <= BACKGROUND_BUDGET + 1e-12
display(pd.Series(policy_metrics, name="policy_partition"))
print("Frozen threshold:", threshold, "| observed background budget:", BACKGROUND_BUDGET)
TEST_EVALUATED = globals().get("_section4_test_claimed",False)


## 7. Optional historical replay
When explicitly enabled, parse 2017–2018 records. Exclude every address seen before 2017, then apply the same deterministic cleaning and first-observation rule. Do not resample, rebalance, retune, adjust cutoffs or change the winner. The resulting test has a different population from all daily corpus rows.

Report ranking, precision/recall, confusion counts, background-alert rate and alert workload. False positives here are alerts on incompletely labelled background. Precision is measured at the corpus's sampled prevalence. No modern deployment, live prevention, monetary risk or zero-day claim follows.


In [ ]:
if RUN_HISTORICAL_TEST:
    if not FROZEN or TEST_EVALUATED:
        raise RuntimeError("Freeze first; final test may be evaluated only once per fresh run")
    _section4_test_claimed = True
    TEST_EVALUATED = True  # guard before parsing; rerun from a fresh runtime after a failed final step
    test, test_audit = read_period(archive_bytes, CUTS[3], pd.Timestamp("2019-01-01"))
    require_binary(test, "final test")
    assert not set(test.address).intersection(development.address)
    assert development.date.max() < test.date.min()
    started = time.perf_counter()
    test_score = winner_scores(test)
    test_score_seconds = time.perf_counter() - started
    test_metrics = operating_metrics(test.y, test_score, threshold)
    display(pd.Series(test_audit, name="test_audit"))
    display(pd.Series(test_metrics, name="historical_test"))
    print("Test score seconds:", test_score_seconds)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    precision, recall, _ = precision_recall_curve(test.y, test_score)
    axes[0].plot(recall, precision, label=winner)
    axes[0].axhline(test.y.mean(), linestyle="--", label="test prevalence")
    axes[0].set(xlabel="Recall", ylabel="Precision", title="Historical test PR curve")
    axes[0].legend()
    ConfusionMatrixDisplay(confusion_matrix(test.y, test_score >= threshold, labels=[0, 1]),
                           display_labels=["background", "known ransomware"]).plot(ax=axes[1], colorbar=False)
    plt.tight_layout()
    plt.show()
else:
    print("Historical test replay disabled; no fresh holdout performance is claimed.")


## 8. Uncertainty, drift and aggregate errors
Addresses are disjoint, but daily graph features can still share transactions and campaigns. Bootstrap complete calendar-week blocks for operating-point intervals, preserving within-week dependence. This assumes weeks approximate independent, representative blocks; longer dependence or temporal drift invalidates that assumption. Intervals describe this historical sample, not deployment guarantees. AP/ROC-AUC are point estimates only. A degenerate interval (for example, zero false alerts in every replicate) cannot bound unobserved errors; inspect the flagged degeneracy rather than reading zero as proof of zero risk.

Check time bins, each labelled family, and families absent from the entire development pool. Small subgroups are descriptive. Drift summaries use fitting quantiles and report missingness separately. They are diagnostics, not tests proving distribution equality.


In [ ]:
def block_intervals(d, score, threshold, repeats=BOOTSTRAPS):
    pred, y = np.asarray(score) >= threshold, d.y.to_numpy()
    weeks = d.date.dt.to_period("W").astype(str)
    rows = pd.DataFrame({"week": weeks, "tn": ((y == 0) & ~pred).astype(int),
                         "fp": ((y == 0) & pred).astype(int), "fn": ((y == 1) & ~pred).astype(int),
                         "tp": ((y == 1) & pred).astype(int)})
    totals = rows.groupby("week", observed=True)[["tn", "fp", "fn", "tp"]].sum().to_numpy()
    if len(totals) < 8:
        return {"status": "too few calendar blocks", "blocks": len(totals)}
    rng, samples = np.random.default_rng(SEED), []
    for _ in range(repeats):
        tn, fp, fn, tp = totals[rng.integers(0, len(totals), len(totals))].sum(axis=0)
        samples.append([tp / (tp + fp) if tp + fp else np.nan,
                        tp / (tp + fn) if tp + fn else np.nan,
                        fp / (fp + tn) if fp + tn else np.nan,
                        (tp + fp) / (tn + fp + fn + tp)])
    result = {"blocks": len(totals), "bootstrap_repeats": repeats, "interval": "percentile 95%, calendar-week blocks"}
    for i, name in enumerate(("precision", "recall", "background_alert_rate", "alert_fraction")):
        vals = np.asarray(samples)[:, i]
        vals = vals[np.isfinite(vals)]
        result[name] = {"lower": float(np.quantile(vals, .025)) if len(vals) else None,
                        "upper": float(np.quantile(vals, .975)) if len(vals) else None,
                        "defined_replicates": len(vals),
                        "degenerate": bool(len(vals) and np.all(vals == vals[0]))}
    return result

def subgroup_table(d, score, threshold, groups):
    rows = []
    groups = np.asarray(groups)
    for group in sorted(set(groups)):
        mask = groups == group
        metrics = operating_metrics(d.y.to_numpy()[mask], np.asarray(score)[mask], threshold)
        rows.append({"group": str(group), **metrics})
    return pd.DataFrame(rows)

def drift_table(train, other):
    rows = []
    for name in FEATURES:
        values = train[name].dropna()
        q1, q99 = values.quantile([.01, .99]) if len(values) else (np.nan, np.nan)
        present = other[name].notna()
        outside = other.loc[present, name].lt(q1) | other.loc[present, name].gt(q99)
        rows.append({"feature": name, "fit_missing": float(train[name].isna().mean()),
                     "test_missing": float(other[name].isna().mean()),
                     "test_outside_fit_1_99_quantiles": float(outside.mean()) if len(outside) else None})
    return pd.DataFrame(rows)


In [ ]:
if RUN_HISTORICAL_TEST:
    intervals = block_intervals(test, test_score, threshold)
    family_metrics = subgroup_table(test, test_score, threshold, test.family)
    time_metrics = subgroup_table(test, test_score, threshold, test.date.dt.to_period("Q").astype(str))
    known_families = set(development.loc[development.y == 1, "family"])
    novelty = np.where(test.y == 0, "background",
                       np.where(test.family.isin(known_families), "development-seen family", "development-unseen family"))
    novelty_metrics = subgroup_table(test, test_score, threshold, novelty)
    drift = drift_table(fitting, test)
    display(intervals)
    display(family_metrics)
    display(time_metrics)
    display(novelty_metrics)
    display(drift)
    # Explain the frozen classical winner on selection data, never tune from importance.
    importance_table = pd.DataFrame()
    if winner in bundles and bundles[winner]["kind"] == "classical" and winner != "prior":
        subset = cap_rows(selection, 10_000)
        if subset.y.nunique() < 2:
            subset = selection  # no class-dependent sampling; full selection has verified support
        bundle = bundles[winner]
        with threadpool_limits(limits=2):
            imp = permutation_importance(bundle["model"], subset[bundle["columns"]], subset.y,
                                         scoring="average_precision", n_repeats=3, random_state=SEED, n_jobs=1)
        importance_table = pd.DataFrame({"feature": bundle["columns"], "selection_ap_decrease": imp.importances_mean,
                                        "repeat_std": imp.importances_std}).sort_values("selection_ap_decrease", ascending=False)
        display(importance_table)
    else:
        print("Permutation importance skipped for prior, ensemble or neural winner; no substitute model is fitted.")
else:
    print("Historical test replay disabled; no fresh holdout performance is claimed.")


## 9. Evidence and interpretation
Export aggregate audit counts, declared configuration, source fingerprint, versions, CV/selection summaries, neural curves, policy/test metrics, intervals and subgroup diagnostics. No addresses, raw rows, trained weights or personal details are exported. The notebook does not publish anything to GitHub.

Write your conclusion from actual outputs: whether the winner beats prevalence, how forward-fold AP varies, whether engineered features help, whether neural cost is justified, and whether the policy's recall/alert tradeoff survives time drift. Never call a model universally best. After inspecting this test, further development needs an independent new holdout. The archive is historical and incomplete labels, graph dependence, first-observation filtering and sampled background all limit interpretation.


In [ ]:
def json_safe(value):
    if isinstance(value,dict): return {str(k):json_safe(v) for k,v in value.items()}
    if isinstance(value,(list,tuple)): return [json_safe(v) for v in value]
    if isinstance(value,np.generic): return json_safe(value.item())
    if isinstance(value,float) and not np.isfinite(value): return None
    return value
def partition_fingerprint(d):
    h=hashlib.sha256()
    for row in d[["address","date","family"]].sort_values(["date","address"]).itertuples(index=False,name=None):
        h.update(json.dumps([str(v) for v in row]).encode()+b"\n")
    return h.hexdigest()
fingerprints={name:partition_fingerprint(d) for name,d in parts.items()}
coverage=pd.DataFrame({"full_fit":fit_pool.groupby("family").size(),
                       "sampled_fit":fitting.groupby("family").size()}).fillna(0).astype(int).rename_axis("family").reset_index()
display(coverage)
policy_family_metrics=subgroup_table(policy,policy_score,threshold,policy.family)
display(policy_family_metrics)
evidence={"scenario":"ransomware-linked payment address screening","version":EXPERIMENT_VERSION,
    "source":provenance,"versions":versions,"runtime_strategy":accelerator,
    "config":{"seed":SEED,"max_fit_addresses":MAX_FIT_ADDRESSES,"max_cv_eval":MAX_CV_EVAL,
              "cutoffs":[str(x.date()) for x in CUTS],"background_budget":BACKGROUND_BUDGET,
              "run_neural":RUN_NEURAL,"neural_seeds":NEURAL_SEEDS,
              "sampling":"all fitting positives plus uniform background",
              "weighting":"inverse inclusion times original-population class balance; prior inclusion only"},
    "winner":winner,"threshold":threshold,"partition_sha256":fingerprints,"fit_seconds":fit_seconds,
    "development_audit":development_audit,"policy_metrics":policy_metrics,
    "neural_histories":neural_histories,"fresh_holdout_available":False,
    "evaluation_status":"development_only_no_fresh_holdout","historical_test_enabled":RUN_HISTORICAL_TEST,
    "limits":["white not verified benign; sampled corpus prevalence",
              "first-observation population; graph and time dependence",
              "retrospective labels; historical benchmark inspected",
              "Further independent improvement requires new compatible data"]}
tables={"cv":cv_table,"selection":selection_table,"fit_family_coverage":coverage,
        "policy_families":policy_family_metrics}
if RUN_HISTORICAL_TEST:
    evidence.update({"historical_test_metrics":test_metrics,"historical_test_audit":test_audit,
        "historical_intervals":intervals,"historical_test_is_independent":False,
        "historical_test_sha256":partition_fingerprint(test),"historical_test_score_seconds":test_score_seconds})
    tables.update({"historical_families":family_metrics,"historical_time":time_metrics,
                   "historical_novelty":novelty_metrics,"historical_drift":drift,"historical_importance":importance_table})
(RESULTS/"evidence.json").write_text(json.dumps(json_safe(evidence),indent=2,allow_nan=False),encoding="utf-8")
for name,table in tables.items(): table.to_csv(RESULTS/(name+".csv"),index=False)
print("Aggregate development evidence:",RESULTS)
print("Frozen winner:",winner,"| policy AP:",policy_metrics["ap"],"| policy recall:",policy_metrics["recall"])
print("No new independent performance is claimed.")
